# VUN sanity check

Loads the cleaned **QM9** and **ZINC-250k** and runs the VUN sanity check.

## Setup

In [1]:
import os

REPO = "flow-matching-molecules"
if not os.path.isdir(REPO):
    !git clone https://github.com/Nico-Conti/flow-matching-molecules.git
os.chdir(REPO if os.path.basename(os.getcwd()) != REPO else ".")

!pip install -q uv
!uv pip install --system -q -e .
print("cwd:", os.getcwd())

Cloning into 'flow-matching-molecules'...
remote: Enumerating objects: 63, done.
remote: Counting objects: 100% (63/63), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 63 (delta 23), reused 55 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (63/63), 31.53 KiB | 1.75 MiB/s, done.
Resolving deltas: 100% (23/23), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.7/24.7 MB 60.1 MB/s eta 0:00:00
cwd: /content/flow-matching-molecules


## 1 · Load cleaned datasets from the Hub

`use_cache=True` pulls the cleaned `(smiles, y)` tables (public, token-free) and skips the
download + sanitize pass entirely.

In [2]:
from dataset.qm9 import load_qm9
from dataset.zinc import load_zinc

qm9 = load_qm9(use_cache=True)
zinc = load_zinc(use_cache=True)
print("QM9 :", qm9["ds"].num_rows, "| ZINC:", zinc["ds"].num_rows)

/usr/local/lib/python3.12/dist-packages/torch_molecule/generator/molgpt/modeling_molgpt.py:112: SyntaxWarning: invalid escape sequence '\['
  self.pattern = "(\[[^\]]+]|<|Br?|Cl?|N|O|S|P|F|I|b|c|n|o|s|p|\(|\)|\.|=|#|-|\+|\\\\|\/|:|~|@|\?|>|\*|\$|\%[0-9]{2}|[0-9])"
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/1.22k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/5.94M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/132040 [00:00<?, ? examples/s]

README.md:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/247449 [00:00<?, ? examples/s]

QM9 : 132040 | ZINC: 247449


## 2 · Dataset-row VUN sanity

Round-trips the **train** molecules through the featurizer (`smiles -> (X, E) ->
tensor_to_mol -> largest_fragment -> SMILES`) and scores VUN on both datasets. Expect
**Valid ≈100%** and **Unique ≈100%**.
Novelty is **0 by construction** (train scored against itself). This is the dataset baseline.

In [3]:
from dataset.featurize import smiles_to_tensor
from dataset.metrics import vun_from_graphs

SANITY_N = None   # subsample for speed; set None for the full dataset

def dataset_row(name, d):
    smis = list(d["ds"]["smiles"])
    sample = smis[:SANITY_N] if SANITY_N else smis
    graphs = [smiles_to_tensor(s, atom_vocab=d["atom_vocab"]) for s in sample]
    m = vun_from_graphs(graphs, train_smiles=smis, atom_vocab=d["atom_vocab"])
    print(f"{name}: valid={m['validity']:.4f}  unique={m['uniqueness']:.4f}  "
          f"(novelty={m['novelty']:.4f}, self-check)  | n={m['n_generated']:,}")

dataset_row("QM9 ", qm9)
dataset_row("ZINC", zinc)

QM9 : valid=1.0000  unique=0.9993  (novelty=0.0000, self-check)  | n=132,040
ZINC: valid=1.0000  unique=0.9933  (novelty=0.0000, self-check)  | n=247,449
